In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors


def load_results(csv_file):
    """Load earthquake catalog and log-likelihood results."""
    data = pd.read_csv(csv_file)
    mask = data["step"].isin(["train", "val", "test"])
    return {
        "time": data.loc[mask, "time"].values,
        "magnitude": data.loc[mask, "magnitude"].values,
        "longitude": data.loc[mask, "longitude"].values,
        "latitude": data.loc[mask, "latitude"].values,
        "ll_t": data.loc[mask, "logli_t"].values,
        "ll_s": data.loc[mask, "logli_s"].values,
        "train_num": int((data["step"] == "train").sum()),
        "val_num": int((data["step"] == "val").sum()),
        "test_num": int((data["step"] == "test").sum()),
    }


def get_year_ticks(time_arr, x_arr, interval=2):
    """Generate x tick positions and labels every `interval` years."""
    positions = []
    labels = []
    seen = set()
    last = None

    for x, t in zip(x_arr, time_arr):
        year = int(str(t).split("-")[0])
        if year in seen:
            continue
        seen.add(year)
        positions.append(x)
        if last is None or year - last >= interval:
            labels.append(str(year))
            last = year
        else:
            labels.append("")
    return positions, labels


def plot_scatter(
    x_arr,
    y_arr,
    c_arr,
    time_arr,
    training_num,
    validation_num,
    label,
    y_label,
):
    """Plot scatter figure with top year axis."""

    cmap = plt.get_cmap("coolwarm")
    vmax = np.percentile(np.abs(c_arr), 90)
    norm = colors.Normalize(vmin=-vmax, vmax=vmax)

    fig, ax = plt.subplots(figsize=(12, 6), dpi=300)
    ax_top = ax.twiny()

    sc = ax.scatter(
        x_arr,
        y_arr,
        c=c_arr,
        cmap=cmap,
        norm=norm,
        s=1,
    )

    ax.axvline(training_num, color="grey", linestyle="--")
    ax.axvline(training_num + validation_num, color="grey", linestyle="--")

    ylim = ax.get_ylim()
    xlim = ax.get_xlim()

    textbox = dict(
        fontsize=10,
        fontfamily="serif",
        bbox=dict(facecolor="white", edgecolor="black",
                  boxstyle="square,pad=0.3", alpha=0.9),
        ha="center",
        va="top",
    )

    ax.text(training_num / 2, ylim[1] * 0.95, "Training Set", **textbox)
    ax.text(training_num + validation_num / 2,
            ylim[1] * 0.95, "Validation Set", **textbox)
    ax.text(
        training_num + validation_num +
        (xlim[1] - training_num - validation_num) / 2,
        ylim[1] * 0.95,
        "Test Set",
        **textbox,
    )

    ax.set_xlabel("Index of Earthquakes", fontsize=14)
    ax.set_ylabel(y_label, fontsize=14)

    ticks, labels = get_year_ticks(time_arr, x_arr)
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xlabel("Time (Year)", fontsize=14)
    ax_top.set_xticks(ticks)
    ax_top.set_xticklabels(labels, rotation=45, fontsize=12)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_linewidth(1.5)
    ax.spines["left"].set_linewidth(1.5)
    ax.tick_params(width=1.5, labelsize=12)

    ax_top.spines["bottom"].set_visible(False)
    ax_top.spines["left"].set_visible(False)
    ax_top.spines["right"].set_visible(False)
    ax_top.spines["top"].set_linewidth(1.5)
    ax_top.tick_params(width=1.5)

    ax.grid(True, linestyle="--", alpha=0.5)

    cbar = fig.colorbar(sc, ax=ax, pad=0.03, aspect=20, extend="both")
    if "spatial" in label:
        cbar.set_label(
            r"$\mathcal{L}_{Spatial,\ ST\!-\!NKF}-\mathcal{L}_{Spatial,\ ETAS}$",
            fontsize=13,
        )
    else:
        cbar.set_label(
            r"$\mathcal{L}_{Temporal,\ ST\!-\!NKF}-\mathcal{L}_{Temporal,\ ETAS}$",
            fontsize=13,
        )

    out_dir = Path("fig")
    out_dir.mkdir(exist_ok=True)

    fig.tight_layout()
    outfile = out_dir / f"IG-{label}-{y_label}.png"
    fig.savefig(outfile, dpi=500, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {outfile}")


In [2]:
datasets = [
    "ComCat",
    "SaltonSea",
    "SanJac",
    "WHITE",
    "SCEDC_20",
    "SCEDC_25",
    "SCEDC_30",
]

for dataset in datasets:
    etas = load_results(
        f"./csv/{dataset}_empirical_empirical_empirical.csv"
    )
    nkf = load_results(
        f"./csv/{dataset}_neural_neural_neural.csv"
    )

    print(
        f"{dataset}: train={etas['train_num']}, "
        f"val={etas['val_num']}, test={etas['test_num']}"
    )

    plot_scatter(
        x_arr=np.arange(1, len(etas["magnitude"]) + 1),
        y_arr=etas["magnitude"],
        c_arr=nkf["ll_t"] - etas["ll_t"],
        time_arr=etas["time"],
        training_num=etas["train_num"],
        validation_num=etas["val_num"],
        label=f"{dataset}_temporal",
        y_label="Magnitude",
    )


ComCat: train=40701, val=14741, test=21888
Saved: fig/IG-ComCat_temporal-Magnitude.png
SaltonSea: train=36639, val=2683, test=4104
Saved: fig/IG-SaltonSea_temporal-Magnitude.png
SanJac: train=11770, val=3449, test=4400
Saved: fig/IG-SanJac_temporal-Magnitude.png
WHITE: train=18548, val=13812, test=24080
Saved: fig/IG-WHITE_temporal-Magnitude.png
SCEDC_20: train=81433, val=23484, test=13071
Saved: fig/IG-SCEDC_20_temporal-Magnitude.png
SCEDC_25: train=25257, val=9089, test=5000
Saved: fig/IG-SCEDC_25_temporal-Magnitude.png
SCEDC_30: train=6815, val=3135, test=1898
Saved: fig/IG-SCEDC_30_temporal-Magnitude.png
